## Import Library

In [2]:
## utama
import pandas as pd
import numpy as np
from textblob import Word

## preprocessing
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from collections import Counter
from nltk.tokenize import sent_tokenize

## data visualization
import matplotlib.pyplot as plt
import seaborn as sns

## model building
from sklearn import svm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold

## model evaluation
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split

## lainnya
from numpy import array
from scipy.sparse import csr_matrix, coo_matrix, hstack
from numpy import mean
from nltk.corpus import words
import time
import csv
import os
from sklearn.preprocessing import StandardScaler

In [ ]:
# os.remove("././confusion_matrix_tree_600.pdf")

## Data Preparation

In [ ]:
## all data

# df_beauty = pd.read_csv('../input/beauty/beauty.csv')
# df_games = pd.read_csv('../input/vidgames/vidgames.csv')

# df_beauty = pd.read_csv('../input/beauty/beauty.csv', nrows=500).drop(['marketplace', 'review_id', 'product_parent', 'product_category', 'star_rating', 'vine', 'review_headline', 'review_date'], axis=1)
# df_games = pd.read_csv('../input/vidgames/vidgames.csv', nrows=500).drop(['marketplace', 'review_id', 'product_parent', 'product_category', 'star_rating', 'vine', 'review_headline', 'review_date'], axis=1)

## all clean review
# master_beauty = pd.read_csv('../input/masterbeauty/masterbeauty.csv', nrows=100).drop(['marketplace', 'review_id', 'product_parent', 'product_category', 'star_rating', 'vine', 'review_headline', 'review_date'], axis=1)
# master_games = pd.read_csv('../input/mastervidgames/mastervidgames.csv', nrows=100).drop(['marketplace', 'review_id', 'product_parent', 'product_category', 'star_rating', 'vine', 'review_headline', 'review_date'], axis=1)

In [3]:
## clean data (sudah diberi label, di preprocess, dan di seimbangkan class nya)

## 1000
master_beauty_1000 = pd.read_csv('../input/reviews/1000_final_beauty.csv')
master_games_1000 = pd.read_csv('../input/reviews/1000_final_games.csv')
## 3000
master_beauty_3000 = pd.read_csv('../input/reviews/3000_final_beauty.csv')
master_games_3000 = pd.read_csv('../input/reviews/3000_final_games.csv')
## 5000
master_beauty_5000 = pd.read_csv('../input/reviews/5000_final_beauty.csv')
master_games_5000 = pd.read_csv('../input/reviews/5000_final_games.csv')
## 8000
master_beauty_8000 = pd.read_csv('../input/reviews/8000_final_beauty.csv')
master_games_8000 = pd.read_csv('../input/reviews/8000_final_games.csv')

In [4]:
contraction_words = pd.read_csv('../input/contraction/contraction_words.csv', header=None, index_col=0, squeeze=True).to_dict()
slang_words = pd.read_csv('../input/slangs/slang_words.csv', header=None, index_col=0, squeeze=True).to_dict()

In [ ]:
# master_beauty_1000.head()

In [ ]:
# print(master_beauty_1000.shape)
# print()
# print(master_beauty_1000.shape)

## Data Preprocessing

In [1]:
## example
second_cleansing(""" Very good, inexpensive brush! Bought 5 for my wife &amp; I. 4 lasted 20 months. On the last one &amp; ready to re-order. Can't beat the price https://www.youtube.com/watch?v=nVHP49g5IPQ. """)

In [5]:
## CUSTOM FUNCTION

def data_labeling(helpful_vote, total_vote):
    if total_vote < 10:
        return 'tidak bermanfaat'
    else:
        ratio_helpfulness = helpful_vote/total_vote
        if ratio_helpfulness > 0.6:
            return 'bermanfaat'
        else:
            return 'tidak bermanfaat'

def second_cleansing(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+", ' ', text) 
    text = re.sub(r"www\S+", ' ', text) 
    text = re.sub(r'<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});', ' ', text) # match html tags
    text = expand_words(text, contraction_words) 
    text = expand_words(text, slang_words)
    text = re.sub(r'[^a-z\s]', ' ', text) # match string a-z only (remove punctuation, symbol, and number)
    text = re.sub(r'\s\s+', ' ', text) # optional => match excess whitespace 
    return text

def expand_words(sentence, expansion_word):
    sentence = sentence.split()
    index_replace = []
    replacement_word = []
    for index, value in enumerate(sentence):        
        for key in expansion_word:
            if value == key:
                index_replace.append(index)
                replacement_word.append(expansion_word[key])
                break
    for index, value_index in enumerate(index_replace):
        sentence[value_index] = replacement_word[index]
    return ' '.join(sentence)

def stopword(sentence):
    string = [i for i in sentence.split() if i not in stopwords.words('english')]
    return ' '.join(string)

wl = WordNetLemmatizer()
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatizer(sentence):
    word_pos_tags = nltk.pos_tag(word_tokenize(sentence)) # get position tag
    word = [wl.lemmatize(tag[0], get_wordnet_pos(tag[1])) for idx, tag in enumerate(word_pos_tags)] # map the position tag and lemmatize the word/token
    return ' '.join(word)

## --------------- spelling correction ---------------
word_list = words.words() 
def word_check(sentence):
    final_word_list = []
    for word in sentence.split():
        if word in word_list:
            final_word_list.append(word)
        else:
            corrected_word = spelling_check(word)
            final_word_list.append(corrected_word)
    return ' '.join(final_word_list) 

def spelling_check(word):
    word = Word(word)
    result = word.spellcheck()
    if result[0][1] == 0:
        corrected_word = spelling_corrector(word)
        return corrected_word
    else:
        return result[0][0]
        
def spelling_corrector(word):
    elongated_word = bool(re.search(r'(.)\1{2,}', word))
    if elongated_word:
        temp_word = word 
        while bool(re.search(r'(.)\1{2,}', temp_word)):
            temp = list(temp_word)
            char = re.search(r'(.)\1{2,}', temp_word)
            temp[char.start():(char.end() - 1)] = ''
            string = "".join(temp)
            word = Word(string)
            result = word.spellcheck()
            if result[0][1] != 0:
                return result[0][0]
                break
            else:
                temp_word = string
                if bool(re.search(r'(.)\1{2,}', temp_word)):
                    continue
                else:
                    return word
                    break
    else:
        return word
## --------------- spelling correction ---------------

def final_preprocess(sentence):
    return word_check(lemmatizer(stopword(second_cleansing(sentence))))

### Data Labeling

In [ ]:
start = time.time()
## ----------------------
df_beauty.loc[:, 'label'] = df_beauty.apply(lambda x: data_labeling(x.helpful_votes, x.total_votes), axis=1)
df_games.loc[:, 'label'] = df_games.apply(lambda x: data_labeling(x.helpful_votes, x.total_votes), axis=1)
## ----------------------
end = time.time()
print(f'{round((end - start), 2)} detik')

In [ ]:
# print(df_beauty['label'].value_counts())
# print()
# print(df_games['label'].value_counts())

### Cleansing 1

In [ ]:
start = time.time()
## ----------------------
## rm empty data ulasan
drop_beauty = df_beauty.dropna(subset=['helpful_votes', 'total_votes', 'verified_purchase', 'review_body'])
drop_games = df_games.dropna(subset=['helpful_votes', 'total_votes', 'verified_purchase', 'review_body'])

## rm ulasan not verified
beauty_purchase = drop_beauty.loc[drop_beauty['verified_purchase'] == 'Y']
games_purchase = drop_games.loc[drop_games['verified_purchase'] == 'Y']

## rm duplicate data
final_beauty = beauty_purchase.drop_duplicates(subset=['customer_id', 'product_id']).reset_index(drop=True)
final_games = games_purchase.drop_duplicates(subset=['customer_id', 'product_id']).reset_index(drop=True)
## ----------------------
end = time.time()
print(f'{round((end - start), 2)} detik')

In [ ]:
# print(final_beauty['label'].value_counts())
# print()
# print(final_games['label'].value_counts())

In [ ]:
# tmp1 = final_beauty[final_beauty['label'] == 'tidak bermanfaat'][:4000]
# tmp2 = final_beauty[final_beauty['label'] == 'bermanfaat'][:4000]
# final_beauty = pd.concat([tmp1, tmp2]).reset_index(drop=True)

# tmp1 = final_games[final_games['label'] == 'tidak bermanfaat'][:4000]
# tmp2 = final_games[final_games['label'] == 'bermanfaat'][:4000]
# final_games = pd.concat([tmp1, tmp2]).reset_index(drop=True)

### Cleansing 2

In [ ]:
start = time.time()
## ----------------------
final_beauty.loc[:, 'clean_review'] = final_beauty.apply(lambda x: final_preprocess(x.review_body), axis=1)
final_games.loc[:, 'clean_review'] = final_games.apply(lambda x: final_preprocess(x.review_body), axis=1)
## ----------------------
end = time.time()
print(f'{round((end - start), 2)} detik')

In [ ]:
# final_beauty.head()

In [ ]:
# final_beauty.to_csv('8000_final_beauty.csv',index=False)
# final_games.to_csv('8000_final_games.csv',index=False)

## Modeling

In [7]:
## CUSTOM FUNCTION

def count_word(sentence):
    sentence = str(sentence)
    clean_sentence = second_cleansing(sentence)
    word_list = str(clean_sentence).split()
    return len(word_list)

def count_sentence(sentence):
    sentence = str(sentence)
    number_of_sentences = sent_tokenize(sentence)
    return len(number_of_sentences)

## ----------------------------------
def modeling(data, feature_name, nrows=None, balance_class=['balance','no_balance']):
    clf = svm.SVC(kernel='rbf')
    kf = KFold(n_splits=10, shuffle=True)
    scores = []
    cm = None

    ## ------
    tes = []
    ## ------
    
    if balance_class == 'balance':
        data1 = data[data['label'] == 'tidak bermanfaat'][:nrows]
        data2 = data[data['label'] == 'bermanfaat'][:nrows]
        data = pd.concat([data1, data2]).reset_index(drop=True)
    elif balance_class == 'no_balance':
        data = data[:nrows]
        
    if feature_name == 'semantic':
        vectorizer = TfidfVectorizer()
        X = data['clean_review'].values.astype('str')
        y = data['label']
        for train_index, test_index in kf.split(X):
            X_train, X_test, y_train, y_test = X[train_index], X[test_index], y[train_index], y[test_index]
            train_vectors = vectorizer.fit_transform(X_train)
            test_vectors = vectorizer.transform(X_test)
            clf.fit(train_vectors, y_train)
            y_pred = clf.predict(test_vectors)
            ## -- confusion matrix
            cm = confusion_matrix(y_test, y_pred, labels=['bermanfaat', 'tidak bermanfaat'])
            ## --
            scores.append( (accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='macro'), recall_score(y_test, y_pred, average='macro'), f1_score(y_test, y_pred, average='macro')) )    
        return get_score(scores), cm
    
    elif feature_name == 'structural':
        data.loc[:, 'jumlah_kata'] = data.apply(lambda x: count_word(x.review_body), axis=1)
        data.loc[:, 'jumlah_kalimat'] = data.apply(lambda x: count_sentence(x.review_body), axis=1)
        X = data[['jumlah_kata', 'jumlah_kalimat']]
        y = data['label']
        for train_index, test_index in kf.split(X):
            X_train, X_test, y_train, y_test = X.iloc[train_index], X.iloc[test_index], y[train_index], y[test_index]
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            ## -- confusion matrix
            cm = confusion_matrix(y_test, y_pred, labels=['bermanfaat', 'tidak bermanfaat'])
            ## --
            scores.append( (accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='macro'), recall_score(y_test, y_pred, average='macro'), f1_score(y_test, y_pred, average='macro')) )    
        return get_score(scores), cm
    
    elif feature_name == 'combined':
        vectorizer = TfidfVectorizer()
        data.loc[:, 'jumlah_kata'] = data.apply(lambda x: count_word(x.review_body), axis=1)
        data.loc[:, 'jumlah_kalimat'] = data.apply(lambda x: count_sentence(x.review_body), axis=1)
        X = data[['clean_review', 'jumlah_kata', 'jumlah_kalimat']]
        y = data['label']
        for train_index, test_index in kf.split(X):
            X_train, X_test, y_train, y_test = X.iloc[train_index], X.iloc[test_index], y[train_index], y[test_index]
            
            ## -- ekstrak semantic 
            train_vectors = vectorizer.fit_transform(X_train['clean_review'].values.astype('str'))
            test_vectors = vectorizer.transform(X_test['clean_review'].values.astype('str'))
            ## -- ekstrak structural
            train_tmp = X_train[['jumlah_kata', 'jumlah_kalimat']]
            sparse_train_tmp = csr_matrix(train_tmp.values)
            test_tmp = X_test[['jumlah_kata', 'jumlah_kalimat']]
            sparse_test_tmp = csr_matrix(test_tmp.values)
            
            combine_train = hstack([train_vectors, sparse_train_tmp])
            combine_test = hstack([test_vectors, sparse_test_tmp])
            
            ## -- standard scaler
            scaling = StandardScaler(with_mean=False)
            latih = scaling.fit_transform(combine_train)
            tes = scaling.transform(combine_test)
            ## -- 
            
            clf.fit(latih, y_train)
            y_pred = clf.predict(tes)
            ## -- confusion matrix
            cm = confusion_matrix(y_test, y_pred, labels=['bermanfaat', 'tidak bermanfaat'])
            ## --
            scores.append( (accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='macro'), recall_score(y_test, y_pred, average='macro'), f1_score(y_test, y_pred, average='macro')) )    
        return get_score(scores), cm


def get_score(scores):
    accuracy = []
    precision = []
    recall = []
    fscore = []
    for score in scores:
        accuracy.append(score[0])
        precision.append(score[1])
        recall.append(score[2])        
        fscore.append(score[3])
    return mean(accuracy), mean(precision), mean(recall), mean(fscore), round(np.std(accuracy), 3) , round(np.std(precision), 3), round(np.std(recall), 3), round(np.std(fscore), 3)

def draw_confusion_matrix(cm, title):
    df_cm = pd.DataFrame(cm, index=['bermanfaat', 'tidak bermanfaat'], columns=['bermanfaat', 'tidak bermanfaat'])
    plt.figure(figsize = (9,6))
    ax = sns.heatmap(df_cm, annot=True, cmap='Blues', fmt='d')
    bottom, top = ax.get_ylim()
    ax.set_ylim(bottom, top)
    plt.title(title)
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.savefig('confusion.pdf', format="pdf")
    plt.show()

### Semantic

In [8]:
be1000 = modeling(master_beauty_1000, 'semantic')
ga1000 = modeling(master_games_1000, 'semantic')

be3000 = modeling(master_beauty_3000, 'semantic')
ga3000 = modeling(master_games_3000, 'semantic')

be5000 = modeling(master_beauty_5000, 'semantic')
ga5000 = modeling(master_games_5000, 'semantic')

be8000 = modeling(master_beauty_8000, 'semantic')
ga8000 = modeling(master_games_8000, 'semantic')

print('1000 data')
print(be1000[0])
print(ga1000[0])
print()

print('3000 data')
print(be3000[0])
print(ga3000[0])
print()

print('5000 data')
print(be5000[0])
print(ga5000[0])
print()

print('8000 data')
print(be8000[0])
print(ga8000[0])
print()

## confusion matrix
cm1 = be1000[1]

1000 data
(0.725, 0.730428016322357, 0.7271235045840982, 0.7228025395951916, 0.051, 0.046, 0.048, 0.052)
(0.8009999999999999, 0.8117235964634141, 0.8027279412980534, 0.7992344494644336, 0.057, 0.055, 0.053, 0.057)

3000 data
(0.7616666666666666, 0.7621686808480665, 0.7614340553697106, 0.7609956197554321, 0.022, 0.022, 0.022, 0.022)
(0.8133333333333332, 0.8157472085192872, 0.8131304034859435, 0.8125081802281697, 0.022, 0.022, 0.023, 0.022)

5000 data
(0.7688, 0.7686425347059795, 0.7690459130661417, 0.7684238301599187, 0.024, 0.024, 0.024, 0.024)
(0.8223999999999998, 0.8231927700560779, 0.8226406225896901, 0.822072950610754, 0.023, 0.022, 0.023, 0.023)

8000 data
(0.7726249999999999, 0.7730847859454762, 0.7726469320987429, 0.7723882673914602, 0.013, 0.013, 0.013, 0.013)
(0.82875, 0.8288950440986994, 0.8287289361894473, 0.8284629920621279, 0.015, 0.015, 0.015, 0.015)



### Structural

In [9]:
be1000 = modeling(master_beauty_1000, 'structural')
ga1000 = modeling(master_games_1000, 'structural')

be3000 = modeling(master_beauty_3000, 'structural')
ga3000 = modeling(master_games_3000, 'structural')

be5000 = modeling(master_beauty_5000, 'structural')
ga5000 = modeling(master_games_5000, 'structural')

be8000 = modeling(master_beauty_8000, 'structural')
ga8000 = modeling(master_games_8000, 'structural')

print('1000 data')
print(be1000[0])
print(ga1000[0])
print()

print('3000 data')
print(be3000[0])
print(ga3000[0])
print()

print('5000 data')
print(be5000[0])
print(ga5000[0])
print()

print('8000 data')
print(be8000[0])
print(ga8000[0])
print()

## confusion matrix
cm2 = be1000[1]

1000 data
(0.7719999999999999, 0.7735709838419512, 0.7727088176177802, 0.7707847525897423, 0.039, 0.039, 0.039, 0.039)
(0.8109999999999999, 0.817772315152881, 0.8105376119928458, 0.8092548868659559, 0.037, 0.04, 0.037, 0.037)

3000 data
(0.7773333333333332, 0.7785113061233122, 0.7795454068303174, 0.7768835819185499, 0.029, 0.028, 0.028, 0.029)
(0.8173333333333334, 0.819669564816069, 0.8175787163943647, 0.8166031837028882, 0.014, 0.014, 0.013, 0.014)

5000 data
(0.78, 0.77975903903404, 0.7799874547023663, 0.7795524920093475, 0.011, 0.011, 0.011, 0.011)
(0.8214, 0.823261245202303, 0.8207724316506037, 0.8205222071393787, 0.013, 0.014, 0.013, 0.013)

8000 data
(0.7802499999999999, 0.7806435112020446, 0.7803917215111923, 0.7801473766303972, 0.013, 0.013, 0.013, 0.013)
(0.8229999999999998, 0.8242532828351632, 0.8230497022308851, 0.8227478794427796, 0.014, 0.014, 0.014, 0.014)



### Combined

In [10]:
be1000 = modeling(master_beauty_1000, 'combined')
ga1000 = modeling(master_games_1000, 'combined')

be3000 = modeling(master_beauty_3000, 'combined')
ga3000 = modeling(master_games_3000, 'combined')

be5000 = modeling(master_beauty_5000, 'combined')
ga5000 = modeling(master_games_5000, 'combined')

be8000 = modeling(master_beauty_8000, 'combined')
ga8000 = modeling(master_games_8000, 'combined')

print('1000 data')
print(be1000[0])
print(ga1000[0])
print()

print('3000 data')
print(be3000[0])
print(ga3000[0])
print()

print('5000 data')
print(be5000[0])
print(ga5000[0])
print()

print('8000 data')
print(be8000[0])
print(ga8000[0])
print()

## confusion matrix
cm3 = be1000[1]

1000 data
(0.69, 0.7110686330409627, 0.6904230897126611, 0.6810757188755988, 0.037, 0.042, 0.036, 0.04)
(0.728, 0.7543559371436195, 0.7289863947742136, 0.7192104297574797, 0.03, 0.024, 0.033, 0.035)

3000 data
(0.712, 0.7160984457351253, 0.7117724124347699, 0.7102060850475265, 0.028, 0.029, 0.028, 0.028)
(0.7573333333333333, 0.7632095113182051, 0.7572559705912477, 0.7552514681779309, 0.021, 0.021, 0.021, 0.022)

5000 data
(0.7208, 0.7245663708538453, 0.7208956412875599, 0.7195500818201663, 0.023, 0.022, 0.022, 0.023)
(0.7734, 0.7774324458235113, 0.7734717794964441, 0.7723358562608112, 0.014, 0.013, 0.012, 0.014)

8000 data
(0.73475, 0.7376577891315246, 0.7348525110417388, 0.733749410574197, 0.017, 0.018, 0.017, 0.017)
(0.7865000000000001, 0.7898881991263995, 0.7867357924376394, 0.7857962208719259, 0.009, 0.009, 0.009, 0.009)



### Confusion Matrix

In [ ]:
# draw_confusion_matrix(cm1, 'semantic')
# draw_confusion_matrix(cm2, 'structural')
# draw_confusion_matrix(cm3, 'combined')

## Example (paper)

### Feature Extraction

In [11]:
## Semantic 

vectorizer = TfidfVectorizer()
temp = master_beauty_1000.copy()
temp = temp[:4]

X = temp['clean_review'].values.astype('str')
y = temp['label']

train_vectors = vectorizer.fit_transform(X) # => kiri sendiri itu index dokumen teks, kanan itu index token, yg ada koma2nya itu hasil pembobotan
token_words = vectorizer.get_feature_names_out()

df = pd.DataFrame(data=train_vectors.toarray(), columns=token_words)
# df = pd.DataFrame(train_vectors.todense().T, index=vectorizer.get_feature_names_out(), columns=[f'D{i+1}' for i in range(len(X))])
# df = pd.DataFrame.sparse.from_spmatrix(train_vectors)

## print results
# df
# print(train_vectors)
# print(train_vectors.getnnz()) 
# print()
# print(train_vectors.toarray())
# print(token_words)
# print(X)

In [ ]:
## Structural

temp = master_beauty_1000.copy()
temp = temp[:4]

temp.loc[:, 'jumlah_kata'] = temp.apply(lambda x: count_word(x.review_body), axis=1)
temp.loc[:, 'jumlah_kalimat'] = temp.apply(lambda x: count_sentence(x.review_body), axis=1)

X = temp[['review_body', 'jumlah_kata', 'jumlah_kalimat']]
y = temp['label']

# X

# -- semantara
# total = X['jumlah_kata'].sum()
# total

In [ ]:
## Kombinasi 
vectorizer = TfidfVectorizer()
temp = master_beauty_1000.copy()
temp = temp[:4]

temp.loc[:, 'jumlah_kata'] = temp.apply(lambda x: count_word(x.review_body), axis=1)
temp.loc[:, 'jumlah_kalimat'] = temp.apply(lambda x: count_sentence(x.review_body), axis=1)

X = temp[['clean_review', 'jumlah_kata', 'jumlah_kalimat']]
y = temp['label']

# --- semantic
train_vectors = vectorizer.fit_transform(X['clean_review'].values.astype('str'))
# --- structural
train_tmp = X[['jumlah_kata', 'jumlah_kalimat']]
sparse_train_tmp = csr_matrix(train_tmp.values)
            
combine_train = hstack([train_vectors, sparse_train_tmp])

token_words = vectorizer.get_feature_names_out()
token_words = np.append(token_words, ['jumlah_kata', 'jumlah_kalimat'])

# pd.DataFrame.sparse.from_spmatrix(sparse_train_tmp)
df = pd.DataFrame(data=combine_train.toarray(), columns=token_words)
df = pd.DataFrame(data=combine_train.toarray())

print(combine_train)
print()
# df
# print(sparse_train_tmp)
# print(sparse_train_tmp)
# print(train_vectors)
# print(token_words)
# train_tmp

# ---- scaling
scaling = StandardScaler(with_mean=False)
hasil1 = scaling.fit_transform(combine_train)
hasil2 = scaling.transform(combine_train)

print(hasil1)
# print()
# print(hasil2)

### ICAST 17th Kumamoto University 2022 Example Preview

In [12]:
# Semantic Feature
vectorizer = TfidfVectorizer()
reviews = [
    "Very good, inexpensive brush! Bought 5 for my wife &amp; I. 4 lasted 20 months. On the last one &amp; ready to re-order. Can't beat the price https://www.youtube.com/watch?v=nVHP49g5IPQ.",
    "AMMMAAAZZZIIINNNGGGGG Don't think twice just buy it. It's amazing how even after the first use what a difference it makes. I love this stuff :)",
    "Who pays 4 dollars more for a $20 gift card?<br />What store doesn't sell gift cards that the extra 4 dollars sounds like a good idea?",
    "Fun game, fast delivery.<br />No problems or complaints.  Nice aqnd fast delivery.  Game is in excellent condition.  Brand New I believe. So it is gr8"
]

clean_reviews = []
for review in reviews:
    clean_reviews.append(final_preprocess(review))

clean_reviews

X = np.asarray(clean_reviews)
train_vectors = vectorizer.fit_transform(X) # => kiri sendiri itu index dokumen teks, kanan itu index token, yg ada koma2nya itu hasil pembobotan
token_words = vectorizer.get_feature_names_out()
df = pd.DataFrame(data=train_vectors.toarray(), columns=token_words)

# type(X)
# print(train_vectors)

In [13]:
# Structural Feature

total_word = []
for review in reviews:
    total_word.append(count_word(review))

# print(total_word)

total_sentences = []
for review in reviews:
    total_sentences.append(count_sentence(review))

# print(total_sentences)

### TF-IDF

In [14]:
corpus = [
    'the house had a tiny little mouse',
    'the cat saw the mouse',
    'the mouse run away from the house',
    'the cat finally ate the mouse',
    'the end of the mouse story'
]

vectorizer = TfidfVectorizer(stop_words='english')
temp = vectorizer.fit_transform(corpus)
# print(temp)
# print(type(temp))
# print()
# print(vectorizer.get_feature_names_out())

In [ ]:
# vectorizer.get_feature_names_out()
# temp.todense() # bentuk array 2D
# tes = pd.DataFrame(temp.todense().T, index=vectorizer.get_feature_names_out(), columns=[f'D{i+1}' for i in range(len(corpus))])
tes = pd.DataFrame(data = temp.toarray(), columns=vectorizer.get_feature_names_out())
tes

### Lainnya

In [15]:
import pandas as pd

groups = [[23,135,3], [123,500,1]]
group_labels = ['views', 'orders']

# Convert data to pandas DataFrame.
df = pd.DataFrame(groups, index=group_labels).T

# Plot.
# pd.concat(
#     [df.mean().rename('average'), df.min().rename('min'), 
#      df.max().rename('max')],
#     axis=1).plot.bar()

In [18]:
label1 = master_beauty_1000['label'].value_counts()[0]
label2 = master_beauty_3000['label'].value_counts()[0]
label3 = master_beauty_5000['label'].value_counts()[0]
label4 = master_beauty_8000['label'].value_counts()[0]

# plotdata = pd.DataFrame({
#     "f1-score":[0.825, 0.823]
#     }, 
#     index=["semantic", "structural"]
# )

# hasil = plotdata.plot(kind="bar")
# plt.ylabel("")
# plt.xlabel("Feature Name")

# plt.savefig('test.jpg')

In [19]:
label1 = master_beauty_1000['label'].value_counts()[0]
label2 = master_beauty_3000['label'].value_counts()[0]
label3 = master_beauty_5000['label'].value_counts()[0]
label4 = master_beauty_8000['label'].value_counts()[0]

plotdata = pd.DataFrame({
    "bermanfaat":[label1, label2, label3, label4],
    "tidak bermanfaat":[label1, label2, label3, label4],
    }, 
    index=["1000", "3000", "5000", "8000"]
)

# hasil = plotdata.plot(kind="bar")
# plt.ylabel("Jumlah Class")
# plt.xlabel("Jumlah Data")

# plt.savefig('test.jpg')

In [20]:
plotdata = pd.DataFrame({
    "pies_2018":[40, 12, 10, 26, 36],
    "pies_2019":[19, 8, 30, 21, 38],
    "pies_2020":[10, 10, 42, 17, 37]
    }, 
    index=["Dad", "Mam", "Bro", "Sis", "Me"]
)
# plotdata.plot(kind="bar")
# plt.title("Mince Pie Consumption Study")
# plt.xlabel("Family Member")
# plt.ylabel("Pies Consumed")